In [ ]:
!pip install torch keras tensorflow
!pip install pydot
!pip install graphviz

# Tiền xử lý dữ liệu

In [3]:
import string
import re
from pickle import dump
from unicodedata import normalize
from numpy import array

# Tải dữ liệu vào bộ nhớ
def load_doc(filename):
	# Mở tệp chỉ ở chế độ đọc
	file = open(filename, mode='rt', encoding='utf-8')
	# Đọc toàn bộ nội dung
	text = file.read()
	# Đóng tệp
	file.close()
	return text

# Tách dữ liệu đã tải thành các câu
def to_pairs(doc):
	lines = doc.strip().split('\n')
	pairs = [line.split('\t') for line in  lines]
	return pairs

# clean danh sách các câu
def clean_pairs(lines):
	cleaned = list()
	# Tạo bảng dịch để loại bỏ dấu câu
	table = str.maketrans('', '', string.punctuation)
	for pair in lines:
		clean_pair = list()
		for line in pair:
			# tokenize dựa trên khoảng trắng
			line = line.split()
			# Chuyển thành chữ thường
			line = [word.lower() for word in line]
			# Loại bỏ dấu câu
			line = [word.translate(table) for word in line]
			# Lưu dưới dạng chuỗi
			clean_pair.append(' '.join(line))
		cleaned.append(clean_pair)
	return array(cleaned)

# Lưu dữ liệu tiền xử lý
def save_clean_data(sentences, filename):
	dump(sentences, open(filename, 'wb'))
	print('Saved: %s' % filename)

# Tải tập dữ liệu
filename = '/content/vie.txt'
doc = load_doc(filename)
# Tách thành các cặp Anh-Việt
pairs = to_pairs(doc)
# Làm sạch các câu
clean_pairs = clean_pairs(pairs)
# Lưu các cặp câu đã làm sạch vào tệp
save_clean_data(clean_pairs, 'english-vie.pkl')
# Kiểm tra một vài cặp câu
for i in range(100):
	print('[%s] => [%s]' % (clean_pairs[i,0], clean_pairs[i,1]))

Saved: english-vie.pkl
[run] => [chạy]
[help] => [giúp tôi với]
[go on] => [tiếp tục đi]
[hello] => [chào bạn]
[hurry] => [nhanh lên nào]
[eat it] => [ăn đi]
[eat it] => [ăn nó đi]
[help me] => [cứu tôi với]
[i agree] => [tôi cũng nghĩ như vậy]
[perfect] => [hoàn hảo]
[we know] => [chúng tôi biết]
[we know] => [chúng ta biết]
[you run] => [bạn chạy]
[cheer up] => [đừng có rầu rĩ quá như thế]
[he tries] => [hắn thử]
[he tries] => [anh thử]
[hurry up] => [thoáng cái chân lên]
[i forgot] => [tôi quên mất rồi]
[im bald] => [tôi bị hói]
[im busy] => [tôi đang bận]
[too late] => [muộn quá]
[i hate tv] => [tôi ghét ti vi]
[i laughed] => [tôi đã cười]
[i laughed] => [tôi cười]
[i will go] => [tôi sẽ đi]
[its cold] => [lạnh]
[its ours] => [đó là của chúng tôi]
[its ours] => [đó là của chúng ta]
[she cried] => [nó đã khóc]
[she cried] => [cô ấy đã khóc]
[sit there] => [hãy ngồi ở đó]
[whats up] => [gì thế]
[whats up] => [có chuyện gì]
[are you ok] => [bạn có sao không]
[find a job] => [hãy tìm m

# Tạo các hàm hỗ trợ

In [4]:
# Tải tập dữ liệu sau tiền xử lý
def load_clean_sentences(filename):
	return load(open(filename, 'rb'))

# Lưu danh sách các câu đã làm tiền xử lý vào tệp
def save_clean_data(sentences, filename):
	dump(sentences, open(filename, 'wb'))
	print('Saved: %s' % filename)

# Huấn luyện tokenizer
def create_tokenizer(lines):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer

# Độ dài tối đa của câu
def max_length(lines):
    return max(len(line.split()) for line in lines)

# Encode và pad sequences
def encode_sequences(tokenizer, length, lines):
    # Mã hóa các chuỗi thành số nguyên
    X = tokenizer.texts_to_sequences(lines)
    # Đệm các chuỗi với giá trị 0
    X = pad_sequences(X, maxlen=length, padding='post')
    return X


# Chia tập dữ liệu để train và test

In [5]:
from pickle import load
from pickle import dump
from numpy.random import rand
from numpy.random import shuffle

# Tải tập dữ liệu
raw_dataset = load_clean_sentences('english-vie.pkl')

# Giảm kích thước tập dữ liệu
n_sentences = 6139
dataset = raw_dataset[:n_sentences, :]

# Trộn ngẫu nhiên tập dữ liệu
shuffle(dataset)

# Chia thành tập huấn luyện và kiểm tra
train, test = dataset[:5000], dataset[1139:]

# Lưu tập dữ liệu
save_clean_data(dataset, 'english-vie-both.pkl')
save_clean_data(train, 'english-vie-train.pkl')
save_clean_data(test, 'english-vie-test.pkl')


Saved: english-vie-both.pkl
Saved: english-vie-train.pkl
Saved: english-vie-test.pkl


# Encoder-Decoder architecture with LSTM layers

## Mô hình LSTM

In [12]:
from pickle import load
from numpy import array
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import plot_model
from keras.models import Sequential
from keras.layers import LSTM, Dense, Embedding, RepeatVector, TimeDistributed
from keras.callbacks import ModelCheckpoint, EarlyStopping

# Optimize GPU memory usage
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Define mô hình dịch máy LSTM không có attention
def define_model(src_vocab, tar_vocab, src_timesteps, tar_timesteps, n_units):
    model = Sequential()
    model.add(Embedding(src_vocab, n_units, input_length=src_timesteps, mask_zero=True))
    model.add(LSTM(n_units))
    model.add(RepeatVector(tar_timesteps))
    model.add(LSTM(n_units, return_sequences=True))
    model.add(TimeDistributed(Dense(tar_vocab, activation='softmax')))
    return model

# Tải các tập dữ liệu
dataset = load_clean_sentences('english-vie-both.pkl')
train = load_clean_sentences('english-vie-train.pkl')
test = load_clean_sentences('english-vie-test.pkl')

# Chuẩn bị tokenizer cho tiếng Anh
eng_tokenizer = create_tokenizer(dataset[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1  # Kích thước từ vựng tiếng Anh
eng_length = max_length(dataset[:, 0])  # Độ dài tối đa câu tiếng Anh
print('English Vocabulary Size:', eng_vocab_size)
print('English Max Length:', eng_length)

# Chuẩn bị tokenizer cho tiếng Việt
vie_tokenizer = create_tokenizer(dataset[:, 1])
vie_vocab_size = len(vie_tokenizer.word_index) + 1  # Kích thước từ vựng tiếng Việt
vie_length = max_length(dataset[:, 1])  # Độ dài tối đa câu tiếng Việt
print('Vietnamese Vocabulary Size:', vie_vocab_size)
print('Vietnamese Max Length:', vie_length)

# Chuẩn bị dữ liệu huấn luyện
trainX = encode_sequences(vie_tokenizer, vie_length, train[:, 1])  # Mã hóa và đệm các câu tiếng Việt
trainY = encode_sequences(eng_tokenizer, eng_length, train[:, 0])  # Mã hóa và đệm các câu tiếng Anh

# Chuẩn bị dữ liệu kiểm tra
testX = encode_sequences(vie_tokenizer, vie_length, test[:, 1])  # Mã hóa và đệm các câu tiếng Việt
testY = encode_sequences(eng_tokenizer, eng_length, test[:, 0])  # Mã hóa và đệm các câu tiếng Anh

# Định nghĩa mô hình
model = define_model(vie_vocab_size, eng_vocab_size, vie_length, eng_length, 256)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Xây dựng mô hình với hình dạng đầu vào cụ thể
model.build((None, vie_length))

# Tóm tắt cấu trúc mô hình
print(model.summary())
plot_model(model, to_file='model.png', show_shapes=True)

# Callbacks
filename = 'model.keras'
checkpoint = ModelCheckpoint(filename, monitor='val_loss', verbose=1, save_best_only=True, mode='min')  # Lưu mô hình tốt nhất
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)  # Dừng sớm khi không cải thiện

# Huấn luyện mô hình
history = model.fit(
    trainX, trainY,
    epochs=100,  # Số lượng vòng lặp huấn luyện
    batch_size=64,
    validation_data=(testX, testY),  # Dữ liệu kiểm tra
    callbacks=[checkpoint, early_stopping],
    verbose=2
)

English Vocabulary Size: 3500
English Max Length: 32
Vietnamese Vocabulary Size: 2282
Vietnamese Max Length: 39


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)              │ (None, 39, 256)             │         584,192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_8 (LSTM)                        │ (None, 256)                 │         525,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector_3 (RepeatVector)       │ (None, 32, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_9 (LSTM)                        │ (None, 32, 256)             │         525,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_4 (TimeDistributed) │ (None, 32, 3500)            │         899,500 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,534,316 (9.67 MB)

 Trainable params: 2,534,316 (9.67 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/100

Epoch 1: val_loss improved from inf to 1.47137, saving model to model.keras
79/79 - 7s - 83ms/step - accuracy: 0.7842 - loss: 2.4412 - val_accuracy: 0.7934 - val_loss: 1.4714
Epoch 2/100

Epoch 2: val_loss improved from 1.47137 to 1.39016, saving model to model.keras
79/79 - 3s - 40ms/step - accuracy: 0.7964 - loss: 1.4474 - val_accuracy: 0.8002 - val_loss: 1.3902
Epoch 3/100

Epoch 3: val_loss improved from 1.39016 to 1.34945, saving model to model.keras
79/79 - 5s - 63ms/step - accuracy: 0.7997 - loss: 1.3852 - val_accuracy: 0.8005 - val_loss: 1.3495
Epoch 4/100

Epoch 4: val_loss improved from 1.34945 to 1.33099, saving model to model.keras
79/79 - 3s - 38ms/step - accuracy: 0.8004 - loss: 1.3534 - val_accuracy: 0.8018 - val_loss: 1.3310
Epoch 5/100

Epoch 5: val_loss improved from 1.33099 to 1.32207, saving model to model.keras
79/79 - 5s - 68ms/step - accuracy: 0.8015 - loss: 1.3409 - val_accuracy: 0.8016 - val_loss: 1.3221
Epoch 6/100

Epoch 6: val_loss improved

## Evaluate LSTM

In [13]:
from pickle import load
from numpy import array
from numpy import argmax
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import load_model
from nltk.translate.bleu_score import corpus_bleu

# Ánh xạ một số nguyên thành một từ
def word_for_id(integer, tokenizer):
	for word, index in tokenizer.word_index.items():
		if index == integer:
			return word
	return None

# Tạo chuỗi đích từ chuỗi nguồn
def predict_sequence(model, tokenizer, source):
	# Dự đoán chuỗi đầu ra từ mô hình
	prediction = model.predict(source, verbose=0)[0]
	# Lấy các chỉ số có xác suất cao nhất
	integers = [argmax(vector) for vector in prediction]
	target = list()
	for i in integers:
		# Chuyển chỉ số thành từ
		word = word_for_id(i, tokenizer)
		if word is None:
			break
		target.append(word)
	return ' '.join(target)

# Đánh giá hiệu suất của mô hình
def evaluate_model(model, tokenizer, sources, raw_dataset):
	actual, predicted = list(), list()
	for i, source in enumerate(sources):
		# Chuyển đổi chuỗi nguồn đã mã hóa
		source = source.reshape((1, source.shape[0]))
		translation = predict_sequence(model, eng_tokenizer, source)
		raw_target, raw_src, _ = raw_dataset[i]
		# In kết quả cho một vài mẫu đầu
		if i < 10:
			print('src=[%s], target=[%s], predicted=[%s]' % (raw_src, raw_target, translation))
		actual.append([raw_target.split()])  # Chuỗi mục tiêu thực tế
		predicted.append(translation.split())  # Chuỗi được dự đoán
	# Tính toán điểm BLEU
	print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
	print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
	print('BLEU-3: %f' % corpus_bleu(actual, predicted, weights=(0.3, 0.3, 0.3, 0)))
	print('BLEU-4: %f' % corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25)))

# Tải các tập dữ liệu
dataset = load_clean_sentences('english-vie-both.pkl')
train = load_clean_sentences('english-vie-train.pkl')
test = load_clean_sentences('english-vie-test.pkl')

# Chuẩn bị tokenizer cho tiếng Anh
eng_tokenizer = create_tokenizer(dataset[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1  # Kích thước từ vựng tiếng Anh
eng_length = max_length(dataset[:, 0])  # Độ dài tối đa câu tiếng Anh

# Chuẩn bị tokenizer cho tiếng Việt
vie_tokenizer = create_tokenizer(dataset[:, 1])
vie_vocab_size = len(vie_tokenizer.word_index) + 1  # Kích thước từ vựng tiếng Việt
vie_length = max_length(dataset[:, 1])  # Độ dài tối đa câu tiếng Việt

# Chuẩn bị dữ liệu huấn luyện và kiểm tra
trainX = encode_sequences(vie_tokenizer, vie_length, train[:, 1])  # Mã hóa dữ liệu huấn luyện
testX = encode_sequences(vie_tokenizer, vie_length, test[:, 1])  # Mã hóa dữ liệu kiểm tra

# Tải mô hình đã huấn luyện
model = load_model('model.keras')

# Kiểm tra trên một số chuỗi trong tập huấn luyện
print('train')
evaluate_model(model, eng_tokenizer, trainX, train)

# Kiểm tra trên một số chuỗi trong tập kiểm tra
print('test')
evaluate_model(model, eng_tokenizer, testX, test)


train
src=[tom nói là anh ấy mong bạn sẽ không làm điều đó], target=[tom said he hopes that you wont do that], predicted=[tom said he hopes hopes wont wont do that]
src=[chiếc đầm của nó trông có vẻ rẻ tiền], target=[her dress looked cheap], predicted=[her dress looked cheap]
src=[thích làm gì thì làm], target=[do whatever you want], predicted=[do what you want]
src=[tôi thích ánh nến], target=[i like candlelight], predicted=[i like candlelight]
src=[bạn nên làm những gì họ bảo bạn làm], target=[youd better do what they ask you to do], predicted=[what better you what you you to to do]
src=[bạn có thực sự nghĩ là tôi có thể làm điều đó không], target=[do you really think i can do that], predicted=[do you really think really i do that]
src=[vì mới bước sang đầu xuân năm mới nên không có nhiều khách lắm], target=[it was early spring so there werent many customers], predicted=[it was early spring spring werent werent many customers]
src=[anh biết tom mà phải không], target=[you know about 

# LSTM with Bahdanau Attention

## Mô hình LSTM có attention

In [11]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Embedding, LSTM, Dense, RepeatVector, TimeDistributed, Add, Activation
from tensorflow.keras.saving import register_keras_serializable  # Import để đăng ký lớp tùy chỉnh

# Lớp Attention của Bahdanau
@register_keras_serializable()  # Đăng ký lớp tùy chỉnh
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        # Thêm trọng số W
        self.W = self.add_weight(name="W", shape=(input_shape[0][-1], input_shape[1][-1]),
                                  initializer="normal", trainable=True)
        # Thêm trọng số b
        self.b = self.add_weight(name="b", shape=(input_shape[1][1],),
                                  initializer="zeros", trainable=True)
        # Thêm trọng số V
        self.V = self.add_weight(name="V", shape=(input_shape[1][1],),
                                  initializer="normal", trainable=True)

    def call(self, inputs):
        query, value = inputs
        value = tf.expand_dims(value, 1)  # Thêm một chiều mới cho value

        # Tính điểm attention
        score = tf.matmul(query, self.W)
        score = tf.matmul(score, value, transpose_b=True)
        score = score + self.b
        score = tf.nn.tanh(score)

        # Reshape self.V trước khi thực hiện nhân ma trận
        score = tf.matmul(score, tf.expand_dims(self.V, 1))

        # Tính trọng số attention
        attention_weights = tf.nn.softmax(score, axis=-1)
        # Tạo vector ngữ cảnh
        context_vector = tf.matmul(attention_weights, value)

        return context_vector, attention_weights

# Định nghĩa mô hình dịch tự động với Attention
def define_model_with_attention(src_vocab, tar_vocab, src_timesteps, tar_timesteps, n_units):
    # Lớp đầu vào
    encoder_inputs = layers.Input(shape=(src_timesteps,))
    decoder_inputs = layers.Input(shape=(tar_timesteps,))

    # Bộ mã hóa (Encoder)
    encoder_embedding = Embedding(src_vocab, n_units)(encoder_inputs)
    encoder_lstm = LSTM(n_units, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
    encoder_states = [state_h, state_c]

    # Bộ giải mã (Decoder)
    decoder_embedding = Embedding(tar_vocab, n_units)(decoder_inputs)
    decoder_lstm = LSTM(n_units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

    # Cơ chế Attention
    attention = Attention()([decoder_outputs, encoder_outputs])
    context_vector, attention_weights = attention

    # Kết hợp vector ngữ cảnh với đầu ra của Decoder
    decoder_combined_context = Add()([decoder_outputs, context_vector])

    # Lớp TimeDistributed với Dense
    decoder_dense = TimeDistributed(Dense(tar_vocab, activation='softmax'))
    outputs = decoder_dense(decoder_combined_context)

    # Định nghĩa mô hình
    model = Model([encoder_inputs, decoder_inputs], outputs)
    return model

# Tải các tập dữ liệu
dataset = load_clean_sentences('english-vie-both.pkl')
train = load_clean_sentences('english-vie-train.pkl')
test = load_clean_sentences('english-vie-test.pkl')

# Chuẩn bị tokenizer cho tiếng Anh
eng_tokenizer = create_tokenizer(dataset[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1
eng_length = max_length(dataset[:, 0])
print('English Vocabulary Size:', eng_vocab_size)
print('English Max Length:', eng_length)

# Chuẩn bị tokenizer cho tiếng Việt
vie_tokenizer = create_tokenizer(dataset[:, 1])
vie_vocab_size = len(vie_tokenizer.word_index) + 1
vie_length = max_length(dataset[:, 1])
print('Vietnamese Vocabulary Size:', vie_vocab_size)
print('Vietnamese Max Length:', vie_length)

# Chuẩn bị dữ liệu huấn luyện
trainX = encode_sequences(vie_tokenizer, vie_length, train[:, 1])
trainY = encode_sequences(eng_tokenizer, eng_length, train[:, 0])

# Chuẩn bị dữ liệu kiểm tra
testX = encode_sequences(vie_tokenizer, vie_length, test[:, 1])
testY = encode_sequences(eng_tokenizer, eng_length, test[:, 0])

# Định nghĩa mô hình với Attention
model = define_model_with_attention(vie_vocab_size, eng_vocab_size, vie_length, eng_length, 256)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# In thông tin mô hình
print(model.summary())
plot_model(model, to_file='model_with_attention.png', show_shapes=True)

# Callbacks
filename = 'model_with_attention.keras'
checkpoint = ModelCheckpoint(filename, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)

# Huấn luyện mô hình
history = model.fit(
    [trainX, trainY], trainY,
    epochs=100,
    batch_size=64,
    validation_data=([testX, testY], testY),
    callbacks=[checkpoint, early_stopping],
    verbose=2
)


English Vocabulary Size: 3500
English Max Length: 32
Vietnamese Vocabulary Size: 2282
Vietnamese Max Length: 39


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2             │ (None, 39)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_3             │ (None, 32)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_3 (Embedding)   │ (None, 39, 256)        │        584,192 │ input_layer_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_4 (Embedding)   │ (None, 32, 256)        │        896,000 │ input_layer_3[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_6 (LSTM)             │ [(None, 256), (None,   │        525,312 │ embedding_3[0][0]      │
│                           │ 256), (None, 256)]     │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_7 (LSTM)             │ [(None, 32, 256),      │        525,312 │ embedding_4[0][0],     │
│                           │ (None, 256), (None,    │                │ lstm_6[0][1],          │
│                           │ 256)]                  │                │ lstm_6[0][2]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention (Attention)     │ [(None, 32, 256),      │         66,048 │ lstm_7[0][0],          │
│                           │ (None, 32, 1)]         │                │ lstm_6[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 32, 256)        │              0 │ lstm_7[0][0],          │
│                           │                        │                │ attention[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_3        │ (None, 32, 3500)       │        899,500 │ add[0][0]              │
│ (TimeDistributed)         │                        │                │                        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 3,496,364 (13.34 MB)

 Trainable params: 3,496,364 (13.34 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/100

Epoch 1: val_loss improved from inf to 1.26904, saving model to model_with_attention.keras
79/79 - 8s - 107ms/step - accuracy: 0.7868 - loss: 1.8627 - val_accuracy: 0.8095 - val_loss: 1.2690
Epoch 2/100

Epoch 2: val_loss improved from 1.26904 to 1.18141, saving model to model_with_attention.keras
79/79 - 4s - 56ms/step - accuracy: 0.8120 - loss: 1.2473 - val_accuracy: 0.8234 - val_loss: 1.1814
Epoch 3/100

Epoch 3: val_loss improved from 1.18141 to 1.01866, saving model to model_with_attention.keras
79/79 - 4s - 46ms/step - accuracy: 0.8354 - loss: 1.1105 - val_accuracy: 0.8463 - val_loss: 1.0187
Epoch 4/100

Epoch 4: val_loss improved from 1.01866 to 0.87700, saving model to model_with_attention.keras
79/79 - 3s - 43ms/step - accuracy: 0.8605 - loss: 0.9568 - val_accuracy: 0.8747 - val_loss: 0.8770
Epoch 5/100

Epoch 5: val_loss improved from 0.87700 to 0.74377, saving model to model_with_attention.keras
79/79 - 3s - 42ms/step - accuracy: 0.8839 - loss: 0.8165 - val

## Đánh giá mô hình thông qua BLEU Score

In [14]:
from tensorflow.keras.models import load_model
import numpy as np

# Tải mô hình đã huấn luyện
model = load_model('model_with_attention.keras', custom_objects={'Attention': Attention})

In [15]:
# Load dữ liệu kiểm tra
test = load_clean_sentences('english-vie-test.pkl')

# Chuẩn bị dữ liệu kiểm tra
testX = encode_sequences(vie_tokenizer, vie_length, test[:, 1])
testY = encode_sequences(eng_tokenizer, eng_length, test[:, 0])

In [16]:
# Dự đoán kết quả
predictions = model.predict([testX, testY], verbose=0)

# Chuyển đổi dự đoán từ dạng one-hot về dạng từ
predicted_sentences = []
for i in range(len(predictions)):
    predicted_seq = predictions[i]
    predicted_words = [eng_tokenizer.index_word[idx] for idx in np.argmax(predicted_seq, axis=1) if idx > 0]
    predicted_sentences.append(' '.join(predicted_words))

In [17]:
from nltk.translate.bleu_score import corpus_bleu

# Chuẩn bị dữ liệu cho corpus_bleu
references = [[sentence.split()] for sentence in test[:, 0]]
candidates = [sentence.split() for sentence in predicted_sentences]

# Tính BLEU score cho toàn bộ tập dữ liệu với các n-gram từ 1 đến 4
bleu_1 = corpus_bleu(references, candidates, weights=(1, 0, 0, 0))
bleu_2 = corpus_bleu(references, candidates, weights=(0.5, 0.5, 0, 0))
bleu_3 = corpus_bleu(references, candidates, weights=(0.33, 0.33, 0.33, 0))
bleu_4 = corpus_bleu(references, candidates, weights=(0.25, 0.25, 0.25, 0.25))

print(f'Corpus BLEU-1 score: {bleu_1:.4f}')
print(f'Corpus BLEU-2 score: {bleu_2:.4f}')
print(f'Corpus BLEU-3 score: {bleu_3:.4f}')
print(f'Corpus BLEU-4 score: {bleu_4:.4f}')

Corpus BLEU-1 score: 0.9880
Corpus BLEU-2 score: 0.9831
Corpus BLEU-3 score: 0.9781
Corpus BLEU-4 score: 0.9689


In [18]:
# Hiển thị một số kết quả dự đoán
for i in range(10):
    print(f'Source: {test[i, 1]}')
    print(f'Target: {test[i, 0]}')
    print(f'Predicted: {predicted_sentences[i]}')
    print('---')

Source: bạn có thể đánh vần họ của bạn dùm tôi
Target: can you spell your last name for me
Predicted: can you spell your last name for me
---
Source: anh ấy đổi tên thành tom jackson
Target: he changed his name to tom jackson
Predicted: he changed his name to tom jackson
---
Source: bạn đã béo hơn trước nhiều rồi đấy
Target: youre much fatter than you used to be
Predicted: youre much fatter than you used to be
---
Source: vui lòng trả quyển sách nếu bạn đã đọc xong
Target: please return the book when youve finished reading it
Predicted: please return the book when youve finished reading it
---
Source: họ đều yêu tôi
Target: they all love me
Predicted: they all love me
---
Source: đó không phải là điều mà tôi đang nghĩ tới
Target: this isnt what i was thinking of
Predicted: this isnt what i was thinking of
---
Source: tom không thể giúp chúng tôi nữa
Target: tom cant help us anymore
Predicted: tom cant help us anymore
---
Source: khí hậu ở nhật ấm như ở trung quốc
Target: the climate of

# So sánh 2 mô hình

In [19]:
import numpy as np
from pickle import load
import tensorflow as tf
from nltk.translate.bleu_score import corpus_bleu
from tensorflow.keras.models import load_model
import time

def load_clean_sentences(filename):
    return load(open(filename, 'rb'))

def evaluate_model(model, test_X, test_Y, tokenizer, name):
    start_time = time.time()

    # Tính loss và accuracy
    if name == "LSTM":
        loss, accuracy = model.evaluate(test_X, test_Y, verbose=0)
        predictions = model.predict(test_X)
    else:
        loss, accuracy = model.evaluate([test_X, test_Y], test_Y, verbose=0)
        predictions = model.predict([test_X, test_Y])

    # Tính perplexity
    perplexity = np.exp(loss)

    # Chuẩn bị dữ liệu cho BLEU score
    predicted_sequences = np.argmax(predictions, axis=-1)
    predicted_texts = []
    reference_texts = []

    for pred, ref in zip(predicted_sequences, test_Y):
        pred_text = tokenizer.sequences_to_texts([pred])[0].split()
        ref_text = tokenizer.sequences_to_texts([ref])[0].split()
        predicted_texts.append(pred_text)
        reference_texts.append([ref_text])

    # Tính BLEU scores
    bleu1 = corpus_bleu(reference_texts, predicted_texts, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(reference_texts, predicted_texts, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(reference_texts, predicted_texts, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(reference_texts, predicted_texts)

    # Số parameters
    trainable_params = model.count_params()

    # Thời gian inference
    inference_time = time.time() - start_time

    return {
        "Model": name,
        "Loss": loss,
        "Accuracy": accuracy,
        "Perplexity": perplexity,
        "BLEU-1": bleu1,
        "BLEU-2": bleu2,
        "BLEU-3": bleu3,
        "BLEU-4": bleu4,
        "Parameters": trainable_params,
        "Inference Time": inference_time
    }

# Load dữ liệu test
test = load_clean_sentences('english-vie-test.pkl')
dataset = load_clean_sentences('english-vie-both.pkl')

# Load tokenizers và chuẩn bị dữ liệu
eng_tokenizer = create_tokenizer(dataset[:, 0])
eng_length = max_length(dataset[:, 0])
vie_tokenizer = create_tokenizer(dataset[:, 1])
vie_length = max_length(dataset[:, 1])

testX = encode_sequences(vie_tokenizer, vie_length, test[:, 1])
testY = encode_sequences(eng_tokenizer, eng_length, test[:, 0])

# Load và đánh giá mô hình
lstm_model = load_model('model.keras')
attention_model = load_model('model_with_attention.keras', custom_objects={'Attention': Attention})

lstm_results = evaluate_model(lstm_model, testX, testY, eng_tokenizer, "LSTM")
attention_results = evaluate_model(attention_model, testX, testY, eng_tokenizer, "LSTM+Attention")

# In kết quả so sánh
print("\nModel Comparison:")
metrics = ["Loss", "Accuracy", "Perplexity", "BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4", "Parameters", "Inference Time"]
for metric in metrics:
    print(f"\n{metric}:")
    print(f"LSTM: {lstm_results[metric]:.4f}")
    print(f"LSTM+Attention: {attention_results[metric]:.4f}")
    print(f"Improvement: {((attention_results[metric] - lstm_results[metric])/lstm_results[metric]*100):.2f}%")

# Lưu kết quả vào file
with open('model_comparison_results.txt', 'w') as f:
    f.write("Model Comparison Results\n")
    f.write("=======================\n\n")
    for metric in metrics:
        f.write(f"{metric}:\n")
        f.write(f"LSTM: {lstm_results[metric]:.4f}\n")
        f.write(f"LSTM+Attention: {attention_results[metric]:.4f}\n")
        f.write(f"Improvement: {((attention_results[metric] - lstm_results[metric])/lstm_results[metric]*100):.2f}%\n\n")

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step

Model Comparison:

Loss:
LSTM: 0.4413
LSTM+Attention: 0.0315
Improvement: -92.87%

Accuracy:
LSTM: 0.9285
LSTM+Attention: 0.9976
Improvement: 7.43%

Perplexity:
LSTM: 1.5548
LSTM+Attention: 1.0320
Improvement: -33.63%

BLEU-1:
LSTM: 0.7201
LSTM+Attention: 0.9880
Improvement: 37.21%

BLEU-2:
LSTM: 0.6269
LSTM+Attention: 0.9831
Improvement: 56.80%

BLEU-3:
LSTM: 0.5533
LSTM+Attention: 0.9781
Improvement: 76.78%

BLEU-4:
LSTM: 0.4816
LSTM+Attention: 0.9689
Improvement: 101.20%

Parameters:
LSTM: 2534316.0000
LSTM+Attention: 3496364.0000
Improvement: 37.96%

Inference Time:
LSTM: 10.0322
LSTM+Attention: 9.0023
Improvement: -10.27%
